# Multi-Agent System

A collaborative AI system where four specialised agents work together to answer queries
using Sarvam `sarvam-m`.

## Architecture

```
User Query
        |
        v
Planner Agent       <- sarvam-m: breaks query into tasks (JSON)
        |
        v
Researcher Agent    <- search tool + sarvam-m synthesis
Executor Agent      <- calculator tool + sarvam-m fallback
        |
        v
Critic Agent        <- sarvam-m: reviews results, writes final answer
        |
        v
Final Answer
```

In [ ]:
%pip install sarvamai>=0.1.26 python-dotenv>=1.0.0

## Setup

Load environment variables, initialise the Sarvam AI client, and configure `sys.path`
so that `agents/` and `tools/` are importable from the notebook.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

# Add recipe root to sys.path so agents/ and tools/ are importable
_RECIPE_ROOT = str(Path.cwd())
if _RECIPE_ROOT not in sys.path:
    sys.path.insert(0, _RECIPE_ROOT)

from agents.planner import plan
from agents.researcher import research
from agents.executor import execute_task
from agents.critic import critique

print("Setup complete.")

## Tools

The agents rely on two stdlib-backed tools. Quick verification that both work correctly.

In [ ]:
from tools.calculator import calculate
from tools.search import search

print("calculator('12 * 8')  ->", calculate("12 * 8"))
print()
print("search('Sarvam AI')   ->")
print(search("Sarvam AI"))

## Agents

### Planner

`plan` sends the query to `sarvam-m` and receives a JSON task list. Each task specifies
which agent should handle it and what input to pass.

In [ ]:
sample_query = "What is 20 percent of 500, and what is Sarvam AI?"
tasks = plan(sample_query, client)

print("Query:", sample_query)
print()
print("Plan:")
print(json.dumps(tasks, indent=2))

### Researcher and Executor

`research` retrieves and summarises information using the search tool and `sarvam-m`.
`execute_task` runs arithmetic first; falls back to `sarvam-m` for non-numeric inputs.

In [ ]:
if tasks:
    for task in tasks[:2]:  # show first two tasks individually
        if task["agent"] == "executor":
            result = execute_task(task, client)
            print(f"Executor result for task {task['task_id']}:")
        else:
            result = research(task, client)
            print(f"Researcher result for task {task['task_id']}:")
        print(result)
        print()